[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_dataloading_bottleneck.ipynb)

# Find a data-loading bottleneck with Hugging Face Trainer

A training job can keep making progress while its GPU repeatedly waits for the next batch. This notebook makes that wait visible, then fixes it without rewriting a training loop.

You will train the same Hugging Face ResNet-50 job twice on the real, full-resolution Imagenette dataset. The only difference is three `TrainingArguments` data-loader settings: workers, pinned memory, and persistent workers. For each run, TraceML reports the diagnosis; the final cell also compares wall-clock time and GPU utilization.

The result is intentionally hardware-dependent. More CPU cores can decode images in parallel and keep the GPU fed; a small Colab CPU will still benefit, but may remain input-bound. That is useful information about *your* machine, not a failed demo.

**Before you start:** in Colab, choose **Runtime → Change runtime type → GPU**, then run the cells from top to bottom. This notebook downloads about 1.5 GB of images and a ResNet-50 checkpoint.

## 1. Check that a GPU is available

In [ ]:
!nvidia-smi -L

import torch

print("CUDA available:", torch.cuda.is_available())
assert (
    torch.cuda.is_available()
), "No GPU found. In Colab: Runtime → Change runtime type → GPU, then rerun."

## 2. Install the notebook dependencies

`TraceMLTrainer` is TraceML's Hugging Face `Trainer` drop-in. The notebook uses standard Hugging Face `TrainingArguments`; there is no custom training loop to learn.

In [ ]:
!pip install -q "traceml-ai[hf]" transformers datasets accelerate

## 3. Download a dataset where image decoding matters

CIFAR-sized images are too small to expose much decoding work. Imagenette contains real JPEGs, so `num_workers=0` makes data loading a realistic source of GPU idle time.

In [ ]:
import os

if not os.path.isdir("imagenette2/train"):
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
    !tar -xzf imagenette2.tgz

print("CPU cores available:", os.cpu_count())
print(
    "Training images:",
    sum(len(files) for _, _, files in os.walk("imagenette2/train")),
)

## 4. The complete Trainer script

This is ordinary Hugging Face training: an AutoModel, `TrainingArguments`, and a `Trainer`-style call to `train()`. The TraceML-specific pieces are deliberately small:

1. Call `traceml_hf.init()` once so TraceML can measure internally-created DataLoader fetches.
2. Replace `Trainer` with `TraceMLTrainer` and set `traceml_enabled=True`.

The `--profile` flag is the experiment switch. Model, images, batch size, augmentation, and number of steps stay fixed.

In [ ]:
%%writefile hf_train.py
import argparse
import os

import torch
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, ToTensor
from transformers import AutoImageProcessor, AutoModelForImageClassification, DefaultDataCollator, TrainingArguments

from traceml_ai.integrations import huggingface as traceml_hf

MODEL_NAME = "microsoft/resnet-50"


class ImagenetteForTrainer(Dataset):
    """Return the field names expected by AutoModelForImageClassification."""

    def __init__(self, root, transform):
        self.images = ImageFolder(root, transform=transform)
        self.classes = self.images.classes

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        pixel_values, label = self.images[index]
        return {"pixel_values": pixel_values, "labels": label}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", choices=["baseline", "optimized"], required=True)
    parser.add_argument("--data-dir", default="imagenette2")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-steps", type=int, default=200)
    args = parser.parse_args()

    torch.manual_seed(42)
    optimized = args.profile == "optimized"
    # The only experimental change: DataLoader settings exposed by TrainingArguments.
    num_workers = min(4, os.cpu_count() or 2) if optimized else 0

    image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    crop_size = image_processor.size.get("height", image_processor.size.get("shortest_edge", 224))
    transform = Compose([
        RandomResizedCrop(crop_size),
        RandomHorizontalFlip(),
        ToTensor(),
        Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
    ])
    train_dataset = ImagenetteForTrainer(
        os.path.join(args.data_dir, "train"), transform=transform
    )

    id2label = {index: name for index, name in enumerate(train_dataset.classes)}
    label2id = {name: index for index, name in id2label.items()}
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(train_dataset.classes),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=f"outputs/{args.profile}",
        per_device_train_batch_size=args.batch_size,
        max_steps=args.max_steps,
        learning_rate=1e-4,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        remove_unused_columns=False,
        dataloader_num_workers=num_workers,
        dataloader_pin_memory=optimized,
        dataloader_persistent_workers=optimized and num_workers > 0,
    )

    print(
        f"[demo] profile={args.profile} workers={num_workers} "
        f"pin_memory={optimized} persistent_workers={optimized and num_workers > 0} "
        f"batch_size={args.batch_size} max_steps={args.max_steps}",
        flush=True,
    )

    traceml_hf.init()  # Required once: enables DataLoader fetch and step timing.
    trainer = traceml_hf.TraceMLTrainer(  # The one-line Trainer replacement.
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DefaultDataCollator(),
        traceml_enabled=True,
    )
    trainer.train()


if __name__ == "__main__":
    main()

## 5. Run the baseline: one process loads every image

With `dataloader_num_workers=0`, the main training process decodes and augments each batch itself. TraceML should report `INPUT-BOUND` when that work leaves the GPU waiting. The run uses 200 steps—enough for a stable diagnosis without turning this into a long training job.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name hf_baseline hf_train.py --args --profile baseline --data-dir imagenette2 --max-steps 200 --batch-size 32

## 6. Run the optimized loader

Same images, model, batch size, and steps. This profile lets up to four CPU workers decode ahead, pins batches for faster transfer, and keeps workers alive after the loader is created. If your CPU can hide the decode work, the verdict can flip to `COMPUTE-BOUND`.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name hf_optimized hf_train.py --args --profile optimized --data-dir imagenette2 --max-steps 200 --batch-size 32

## 7. Read the before/after result

TraceML writes a portable `final_summary.json` for each run. This cell prints the same evidence you would use on your own Trainer job: total wall time, average GPU utilization, data-loader wait, and TraceML's primary verdict.

In [ ]:
import json
from pathlib import Path


def read_run(name):
    summary = json.loads(Path(f"logs/{name}/final_summary.json").read_text())
    system = summary.get("system", {}).get("global", {}).get("average", {})
    step_time = (
        summary.get("step_time", {}).get("global", {}).get("average", {})
    )
    diagnosis = summary.get("primary_diagnosis", {})
    return {
        "wall_s": summary.get("duration_s"),
        "gpu_util": system.get("gpu_util_percent"),
        "input_wait_ms": step_time.get("input_wait_ms"),
        "verdict": diagnosis.get("status", "NO VERDICT"),
    }


baseline = read_run("hf_baseline")
optimized = read_run("hf_optimized")


def shown(value, suffix=""):
    return "n/a" if value is None else f"{value:.1f}{suffix}"


print(f"{'metric':<22}{'workers = 0':>18}{'workers = up to 4':>20}")
print("-" * 60)
print(
    f"{'wall clock':<22}{shown(baseline['wall_s'], ' s'):>18}{shown(optimized['wall_s'], ' s'):>20}"
)
print(
    f"{'GPU utilization':<22}{shown(baseline['gpu_util'], '%'):>18}{shown(optimized['gpu_util'], '%'):>20}"
)
print(
    f"{'input wait / step':<22}{shown(baseline['input_wait_ms'], ' ms'):>18}{shown(optimized['input_wait_ms'], ' ms'):>20}"
)
print(
    f"{'TraceML verdict':<22}{baseline['verdict']:>18}{optimized['verdict']:>20}"
)

if baseline["wall_s"] and optimized["wall_s"]:
    speedup = baseline["wall_s"] / optimized["wall_s"]
    print(f"\nResult: {speedup:.2f}x faster for the same 200 optimizer steps.")

if optimized["verdict"] == "COMPUTE-BOUND":
    print(
        "Your loader now hides enough decode work that the GPU is the limiting resource."
    )
elif optimized["verdict"] == "INPUT-BOUND":
    print(
        "Workers reduced the wait, but your CPU or storage still cannot fully feed the GPU."
    )
    print(
        "That points to more CPU capacity, faster storage, lighter augmentation, or further loader tuning."
    )
else:
    print(
        "Read the TraceML summary above: the final bottleneck depends on this runtime's hardware."
    )

## Use this in your own Trainer

The experiment is a pattern, not a special ResNet trick. In an existing Hugging Face script, call `traceml_hf.init()` once, replace `Trainer` with `TraceMLTrainer`, and pass `traceml_enabled=True`. Then run it through `traceml run --mode summary ...`.

If TraceML says `INPUT-BOUND`, start by testing `dataloader_num_workers`, `dataloader_pin_memory`, and `dataloader_persistent_workers` one change at a time. Keep the change only when the measured wall time and input wait improve on the hardware that will actually run your job.